# Road Damage Assessment — Colab PRO Setup

Notebook này chạy **Streamlit web dashboard** trên Google Colab PRO (GPU) + tạo public URL qua cloudflared tunnel.

## ⚠️ Cấu hình Runtime type (BẮT BUỘC trước khi chạy)

**Menu > Runtime > Change runtime type:**
- **Runtime type:** Python 3
- **Hardware accelerator:** `T4 GPU` ⚠️ (KHÔNG chọn CPU — inference sẽ chậm 20x)
- **High-RAM:** `On` ✅ (cần cho video dài + batch ảnh)
- **Runtime version:** Latest (recommended)

> Lý do chọn T4: quota Colab Pro cao nhất (16GB VRAM, đủ cho YOLOv12s + FastSAM). A100/H100 thường quota thấp/greyed out.

**Cảnh báo:** Colab session tạm thời (idle timeout ~90 phút, giới hạn 24h). Luôn có bản backup local (`streamlit run app.py` trên laptop).

**Ưu tiên dùng Colab cho:**
- GPU inference nhanh (T4 ~30-50ms thay vì 896ms CPU)
- Render video demo dài (GPU xử lý 17967 frame trong vài phút)
- Train T3 (YOLOv12s-seg trên RDD2022)

**Debug:** Tab `📜 Console` trong dashboard hiển thị log runtime + system info (CUDA, GPU memory).

## 1. Cài đặt dependencies

In [ ]:
# Install Streamlit + cloudflared + ML deps
!pip install streamlit ultralytics opencv-python-headless supervision Pillow matplotlib huggingface-hub -q

# Verify ffmpeg (REQUIRED for browser-playable H264 video re-encode). Pre-installed on Colab.
import shutil, subprocess
if shutil.which('ffmpeg'):
    print('✅ ffmpeg present:', subprocess.run(['ffmpeg','-version'], capture_output=True, text=True).stdout.splitlines()[0])
else:
    print('⚠️ ffmpeg missing — installing (needed for video playback/download)...')
    !apt-get -qq install -y ffmpeg
    print('✅ ffmpeg installed' if shutil.which('ffmpeg') else '❌ ffmpeg install FAILED — video may not play in browser')

# Verify GPU (T4 from Runtime type config)
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    gpu_name = torch.cuda.get_device_name(0)
    if 'T4' in gpu_name:
        print('✅ T4 GPU detected — optimal for Colab Pro')
    elif 'A100' in gpu_name:
        print('✅ A100 GPU detected — premium (may have lower quota)')
    elif 'L4' in gpu_name:
        print('✅ L4 GPU detected — newer, fast')
    else:
        print(f'⚠️ Unknown GPU: {gpu_name}')
else:
    print('❌ WARNING: No GPU! Go to Runtime > Change runtime type > Hardware accelerator: T4 GPU')
    print('   Also enable High-RAM toggle for video processing')

## 2. Git clone source code

Repo: https://github.com/dinhhung893/road-damage-dashboard (public)

Repo chứa: app.py, engine_bridge.py, render_video.py, dumps_src/ (engine + PCI data).
Models download riêng ở cell 3 (từ HuggingFace).

In [ ]:
# Git clone from GitHub
!git clone https://github.com/dinhhung893/road-damage-dashboard.git /content/repo
ADAPTIVE = '/content/repo'
import os; os.chdir(ADAPTIVE)
print(f'ADAPTIVE = {ADAPTIVE}')
print(f'Files: {os.listdir(ADAPTIVE)}')

## 2.5. Mount Google Drive (BẮT BUỘC nếu dùng source Drive)

Chạy cell này để mount Drive. Colab sẽ hiện popup xác thực → cho phép truy cập.

Sau khi mount, dashboard có thể đọc ảnh/video trực tiếp từ Drive (source `📁 Google Drive`).

In [ ]:
# Mount Google Drive — phải chạy trong Colab cell (không thể từ Streamlit)
from google.colab import drive
drive.mount('/content/drive')
import os
print(f'Drive mounted: {os.path.exists("/content/drive/MyDrive")}')
if os.path.exists('/content/drive/MyDrive'):
    print(f'MyDrive contents: {os.listdir("/content/drive/MyDrive")[:10]}')

## 3. Set DUMPS_ROOT + download model weights + test import

Engine source (src/engine/, src/utils/, data/pci_astm_d6433.json) đã có trong repo (dumps_src/).
Chỉ cần download model weights YOLOv12s (~19MB) từ HuggingFace.

In [ ]:
# Set DUMPS_ROOT + download model + test import (chạy TRƯỚC khi start Streamlit)
import os, sys, subprocess
os.chdir(ADAPTIVE)
sys.path.insert(0, ADAPTIVE)

# DUMPS_ROOT = dumps_src/ in repo (has src/engine/ + data/pci_astm_d6433.json)
DUMPS = f'{ADAPTIVE}/dumps_src' if os.path.exists(f'{ADAPTIVE}/dumps_src') else '/content/dumps'
os.environ['DUMPS_ROOT'] = DUMPS
os.makedirs(f'{DUMPS}/models/yolo-rdd2022-benchmark', exist_ok=True)
os.makedirs(f'{DUMPS}/data/samples', exist_ok=True)
print(f'DUMPS_ROOT = {DUMPS}')

# Download YOLOv12s detection weights from HuggingFace (~19MB)
det_model = f'{DUMPS}/models/yolo-rdd2022-benchmark/yolo12s_seed0_best.pt'
if not os.path.exists(det_model):
    print('Downloading YOLOv12s weights (~19MB from HuggingFace)...')
    subprocess.run(['wget', '-q', '-O', det_model,
                    'https://huggingface.co/SreekarAditya/yolo-rdd2022-benchmark/resolve/main/yolo-rdd2022-benchmark/yolo12s_seed0_best.pt'])
if os.path.exists(det_model):
    print(f'✅ Detection model: {os.path.getsize(det_model)/1e6:.1f} MB')
else:
    print('❌ Detection model missing — check wget output')

# Verify engine source + PCI data
print(f'Engine source: {os.path.exists(f"{DUMPS}/src/engine/detector.py")}')
print(f'PCI data: {os.path.exists(f"{DUMPS}/data/pci_astm_d6433.json")}')

# Test full engine_bridge import
import engine_bridge as eb
print(f'Dumps root: {eb._DUMPS_ROOT}')
print(f'Model exists: {eb.DEFAULT_MODEL_PATH.exists()}')
print(f'PCI data exists: {eb.DEFAULT_PCI_DATA.exists()}')
if not eb.DEFAULT_MODEL_PATH.exists():
    raise RuntimeError('Model not found — cannot start Streamlit. Re-run this cell.')

## 4. Chạy Streamlit + cloudflared tunnel

Chạy cell dưới → đợi ~10s → click URL public (xxxx.trycloudflare.com).

In [ ]:
# Download cloudflared binary
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('cloudflared installed')

In [ ]:
# Run Streamlit in background (NO runOnSave — prevents watchdog feedback loop with app.log)
# Auto-pull watchdog handles code sync. After pull, manually refresh browser to see changes.
import subprocess, os
os.chdir(ADAPTIVE)

# Kill any existing streamlit
!pkill -f streamlit 2>/dev/null; sleep 1

# Start streamlit with debug logging — stderr redirected to /content/streamlit_debug.log
# This captures Streamlit backend logs (widget reruns, errors, widget states)
debug_log = '/content/streamlit_debug.log'
debug_file = open(debug_log, 'w')  # will capture stderr
proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.runOnSave', 'false',
     '--browser.gatherUsageStats', 'false',
     '--logger.level', 'debug'],
    stdout=subprocess.PIPE, stderr=debug_file, text=True
)
print(f'Streamlit PID: {proc.pid}')
print(f'Debug log: {debug_log}')

# Wait for streamlit to start
import time
time.sleep(5)
print('Streamlit started on port 8501 (runOnSave=false — manual refresh after git pull)')

In [ ]:
# Start cloudflared tunnel → get public URL
import subprocess, re, time

tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

# Capture the URL from stderr (cloudflared prints it there)
url = None
for _ in range(30):
    line = tunnel.stderr.readline()
    if line:
        print(line.strip())
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            url = m.group(0)
            break
    time.sleep(1)

if url:
    print(f'\n=== PUBLIC URL ===')
    print(f'🌐 {url}')
    print(f'\nMở URL này trong browser. Dashboard chạy trên GPU Colab.')
    print(f'URL sống cho đến khi Colab session kết thúc.')
else:
    print('URL not found in cloudflared output. Check stderr above.')

## 4.5. Auto-pull watchdog (TỰ ĐỘNG sync code từ GitHub)

Cell này chạy nền, `git pull` mỗi 10s. Khi tôi sửa code local + push → Colab tự lấy → Streamlit auto-reload.

**Workflow:** Tôi sửa → `git push` → cell này pull → Streamlit reload → bạn refresh browser.

**Đọc log:** Cell ở dưới sẽ print log mới nhất sau mỗi pull.

In [ ]:
# Auto-pull watchdog — git pull every 10s, print log if changed
import subprocess, time, os, threading
os.chdir(ADAPTIVE)

def git_pull_loop():
    last_hash = ''
    while True:
        try:
            # Fetch first, then reset --hard to avoid divergent branch conflicts
            # (log sync cell may create local commits that diverge from GitHub)
            subprocess.run(['git', 'fetch', 'origin', 'master'], cwd=ADAPTIVE, capture_output=True, timeout=10)
            local_hash = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, cwd=ADAPTIVE).stdout.strip()[:8]
            remote_hash = subprocess.run(['git', 'rev-parse', 'origin/master'], capture_output=True, text=True, cwd=ADAPTIVE).stdout.strip()[:8]
            if remote_hash != local_hash and remote_hash:
                # Reset to remote — discards local commits (log sync etc.)
                subprocess.run(['git', 'reset', '--hard', 'origin/master'], cwd=ADAPTIVE, capture_output=True, timeout=10)
                new_hash = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, cwd=ADAPTIVE).stdout.strip()[:8]
                if new_hash != last_hash:
                    print(f'[{time.strftime("%H:%M:%S")}] ✅ Synced to {new_hash} (reset --hard)')
                    last_hash = new_hash
        except Exception as e:
            print(f'[{time.strftime("%H:%M:%S")}] ⚠️ Sync error: {e}')
        time.sleep(10)

thread = threading.Thread(target=git_pull_loop, daemon=True)
thread.start()
print('🔄 Auto-pull watchdog started — git pull every 10s')
print('💡 Tôi sửa code + push → Colab tự pull → refresh browser (runOnSave=false)')

## 4.6. Log server (cho Devin đọc log từ xa — KHÔNG cần GitHub token)

Colab không thể `git push` lên GitHub nếu thiếu Personal Access Token → đó là lý do log không lên được repo.

Giải pháp: cell dưới chạy 1 **HTTP server + cloudflared tunnel thứ 2** phục vụ file log. Chạy xong nó in ra 1 URL dạng `https://xxxx.trycloudflare.com/app_unified.log`.

**Gửi URL đó cho Devin** → Devin đọc log real-time bất cứ lúc nào (auth-free, ổn định như tunnel của dashboard).

In [ ]:
# Log server — serve the unified log over a SECOND cloudflared tunnel (NO GitHub auth).
# Colab can't push to GitHub without a token, so we expose the log via http.server + tunnel.
# Devin Local reads it by fetching  <URL>/app_unified.log
import subprocess, time, os, threading, re, shutil

LOGDIR = '/content/logserve'
os.makedirs(LOGDIR, exist_ok=True)
LOG_SOURCES = ['/content/app_unified.log', '/content/streamlit_debug.log']

# Mirror live logs into the served dir every 5s
def _mirror():
    while True:
        for s in LOG_SOURCES:
            if os.path.exists(s):
                try:
                    shutil.copy(s, os.path.join(LOGDIR, os.path.basename(s)))
                except Exception:
                    pass
        time.sleep(5)
threading.Thread(target=_mirror, daemon=True).start()

# Restart static server on :9999
subprocess.run(['pkill', '-f', 'http.server 9999'], capture_output=True)
time.sleep(1)
subprocess.Popen(['python', '-m', 'http.server', '9999', '--directory', LOGDIR],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)

# Second cloudflared tunnel for the log server
tun = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:9999'],
                       stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
url = None
for _ in range(30):
    line = tun.stderr.readline()
    if line:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            url = m.group(0)
            break
    time.sleep(1)

if url:
    print('=== LOG SERVER URL — gửi link này cho Devin ===')
    print(f'{url}/app_unified.log')
    print(f'{url}/streamlit_debug.log')
else:
    print('Log tunnel failed — re-run this cell (cloudflared may need a moment).')

## 6. (Tùy chọn) Render video demo với GPU

GPU Colab render video nhanh gấp 20x CPU. Dùng cho video dài.

In [ ]:
# Render video demo với GPU (cuda)
# Upload video của bạn vào Colab (panel bên trái > Files > Upload) trước khi chạy cell này
import os
os.chdir(ADAPTIVE)
os.makedirs(f'{ADAPTIVE}/outputs', exist_ok=True)

# Đường dẫn video input — đổi theo file bạn upload
INPUT_VIDEO = '/content/sample_video.mp4'  # ← đổi thành tên file bạn upload
OUTPUT_VIDEO = f'{ADAPTIVE}/outputs/demo_colab.mp4'

if os.path.exists(INPUT_VIDEO):
    !python render_video.py --input "{INPUT_VIDEO}" --output "{OUTPUT_VIDEO}" --device cuda --stride 5 --max-frames 200 --confidence 0.15
else:
    print(f'⚠️ {INPUT_VIDEO} not found. Upload video to Colab first (Files panel > Upload).')
    print('Or skip this cell — dashboard already works without pre-rendered video.')

In [ ]:
# Download rendered video (nếu đã render ở cell trên)
import os
OUTPUT_VIDEO = f'{ADAPTIVE}/outputs/demo_colab.mp4'
if os.path.exists(OUTPUT_VIDEO):
    from google.colab import files
    files.download(OUTPUT_VIDEO)
else:
    print(f'{OUTPUT_VIDEO} not found. Run render cell above first.')

## 7. Dừng & dọn dẹp

Khi xong, dừng Streamlit + tunnel để giải phóng Colab.

In [ ]:
# Stop everything
!pkill -f streamlit 2>/dev/null
!pkill -f cloudflared 2>/dev/null
print('Stopped Streamlit + cloudflared tunnel')

## Troubleshooting

- **URL không hiện:** Chạy lại cell cloudflared. Đôi khi cần 2-3 lần.
- **Streamlit lỗi import engine_bridge:** Verify cell 3 (DUMPS_ROOT) đã chạy + `Model exists: True`.
- **Model not found:** Re-run cell 3 (download model từ HuggingFace).
- **No GPU:** Runtime > Change runtime type > T4 GPU (Colab PRO).
- **Session ngắt:** Colab PRO idle timeout ~90 phút. Tương tác thường xuyên.
- **Code cũ sau khi tôi push:** Re-run cell auto-pull (3.5) hoặc `!cd {ADAPTIVE} && git pull`.
- **CUDA out of memory:** Giảm max_frames trong Tab Video, hoặc tăng stride.

## Plan B — Local backup

Nếu Colab ngắt giữa demo:
```bash
# Trên laptop (Windows)
cd D:\Antigravity\New folder\adaptive
D:\Antigravity\New folder\dumps\.venv\Scripts\python.exe -m streamlit run app.py
```
Mở http://localhost:8501. Inference CPU ~896ms nhưng tin cậy.